In [1]:
from pathlib import Path
import json
OUT=Path(r'C:/Users/choud/Desktop/Minor project/bilstm_eew/bilstm_training_v1')
n=json.loads((OUT/'01_bilstm_training.ipynb').read_text());exec(compile(''.join(n['cells'][0]['source']),'common_definitions_only','exec'))
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
assert not (OUT/'final_bilstm_eew_report.md').exists(),'Refuse overwrite'
cks={h:torch.load(OUT/'models'/f'bilstm_{h}s_best.pt',map_location='cpu',weights_only=False) for h in [1,2,3]}
assert all(sha(Path(p))==v for p,v in cks[1]['input_hashes'].items())
reference=json.loads(CNN.read_text())['common_cohort'];results={};pdata={}


Device cuda parameters 37730


In [2]:
for h,ck in cks.items():
 assert ck['horizon']==h
 model=TabularBiLSTM().to(DEV);model.load_state_dict(ck['state_dict'],strict=True);model.eval()
 va=load('validation',h);vloss=epoch(model,batches(va),ck['center'],ck['scale'])
 assert abs(vloss['total']-ck['best_validation_loss'])<1e-6,'Checkpoint validation mismatch'
 d=load('test',h);prob,z=infer(model,d['x']);pred=z*ck['scale']+ck['center'] # Single frozen test pass, reused for all subsets.
 y=d['y'].numpy();reg=d['reg'].numpy();target=d['m'].numpy();common=d['common'];cr=reg&common
 digest=hashlib.sha256('\n'.join(d['source'][common]).encode()).hexdigest()
 assert digest==reference['identity_sha256'] and int(cr.sum())==1200 and int((common&~reg).sum())==1214,'Common cohort mismatch'
 a=target[reg];p=pred[reg];error=p-a;bands={}
 for label,mask in [('M<3',a<3),('3<=M<4',(a>=3)&(a<4)),('4<=M<5',(a>=4)&(a<5)),('M>=5',a>=5)]:
  bands[label]={'n':int(mask.sum()),'mae':float(np.abs(error[mask]).mean()),'bias':float(error[mask].mean()),'underprediction_percent':float((error[mask]<0).mean()*100)}
 sample=d['x'][:1].to(DEV)
 with torch.inference_mode():
  for _ in range(25):model(sample)
  if DEV.type=='cuda':torch.cuda.synchronize()
  lat=[]
  for _ in range(200):
   start=time.perf_counter();model(sample)
   if DEV.type=='cuda':torch.cuda.synchronize()
   lat.append((time.perf_counter()-start)*1000)
 selected=detection(y,prob,ck['threshold']);fixed=detection(y,prob,.5);md=magnitude(a,p)
 csel=detection(y[common],prob[common],ck['threshold']);cfix=detection(y[common],prob[common],.5);cmag=magnitude(target[cr],pred[cr]);cnn=reference['metrics'][str(h)]
 results[h]={'threshold':ck['threshold'],'eligible':len(y),'earthquakes':int(reg.sum()),'excluded':d['excluded'],'detection':selected,'fixed_05_detection':fixed,'magnitude':md,'bands':bands,'common':{'detection':csel,'fixed_05_detection':cfix,'magnitude':cmag,'identity_sha256':digest},'cnn_common':cnn,'common_bilstm_minus_cnn':{'fixed_05_accuracy':cfix['accuracy']-cnn['accuracy'],'fixed_05_f1':cfix['f1']-cnn['f1'],'mae':cmag['mae']-cnn['mae'],'rmse':cmag['rmse']-cnn['rmse'],'r2':cmag['r2']-cnn['r2']},'latency_ms':{'median':float(np.median(lat)),'p95':float(np.quantile(lat,.95)),'warmups':25,'repeats':200,'batch_size':1},'validation_test_gaps':{'f1_test_minus_val':selected['f1']-ck['validation_detection']['f1'],'mae_test_minus_val':md['mae']-ck['validation_magnitude']['mae']},'noise_family_shortcuts':'UNKNOWN: no noise_profile field in validated feature file. No invented family labels or expanded source access.'}
 pdata[h]={'a':a,'p':p,'error':error}
 print(f'{h}s eligible F1 {selected["f1"]:.4f}, MAE {md["mae"]:.4f}; common fixed-.5 F1 {cfix["f1"]:.4f}, MAE {cmag["mae"]:.4f}',flush=True)


1s eligible F1 0.7294, MAE 0.6780; common fixed-.5 F1 0.7274, MAE 0.6702
2s eligible F1 0.7285, MAE 0.6383; common fixed-.5 F1 0.6466, MAE 0.6341
3s eligible F1 0.7316, MAE 0.6151; common fixed-.5 F1 0.6947, MAE 0.6151


In [3]:
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
def save(fig,name):
 p=OUT/'plots'/name;assert not p.exists();fig.tight_layout();fig.savefig(p,dpi=300,bbox_inches='tight');plt.close(fig)
fig,axs=plt.subplots(1,3,figsize=(14,4))
for ax,(h,ck) in zip(axs,cks.items()):
 hist=ck['history'];e=[q['epoch'] for q in hist];ax.plot(e,[q['train']['total'] for q in hist],label='Train epoch-end');ax.plot(e,[q['validation']['total'] for q in hist],label='Validation');ax.axvline(ck['best_epoch'],ls=':',color='black');ax.set(title=f'{h}s observed post-P',xlabel='Epoch',ylabel='BCE + masked standardized Huber');ax.legend()
save(fig,'training_validation_loss.png')
fig,axs=plt.subplots(1,3,figsize=(14,4))
for ax,(h,d) in zip(axs,pdata.items()):
 ax.scatter(d['a'],d['p'],s=9,alpha=.35);lo=min(d['a'].min(),d['p'].min());hi=max(d['a'].max(),d['p'].max());ax.plot([lo,hi],[lo,hi],'k--');ax.set(title=f'{h}s: MAE {results[h]["magnitude"]["mae"]:.3f}',xlabel='Actual magnitude',ylabel='Predicted magnitude')
save(fig,'actual_predicted_magnitude.png')
fig,axs=plt.subplots(1,3,figsize=(14,4))
for ax,(h,d) in zip(axs,pdata.items()):ax.hist(d['error'],bins=35);ax.axvline(0,color='black',ls='--');ax.set(title=f'{h}s',xlabel='Predicted minus actual',ylabel='Earthquake count')
save(fig,'residual_distribution.png')
fig,axs=plt.subplots(1,3,figsize=(12,4))
for ax,(h,r) in zip(axs,results.items()):
 cm=np.asarray(r['detection']['confusion_matrix']);ax.imshow(cm,cmap='Blues');ax.set(title=f'{h}s; threshold {r["threshold"]:.2f}',xticks=[0,1],yticks=[0,1],xticklabels=['Noise','EQ'],yticklabels=['Noise','EQ'],xlabel='Predicted',ylabel='Actual')
 for i in range(2):
  for j in range(2):ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black')
save(fig,'confusion_matrices.png')
for kind,name in [('f1','detection_f1_comparison.png'),('mae','magnitude_mae_comparison.png')]:
 fig,ax=plt.subplots(figsize=(8,4));xx=np.arange(3);cnn=[results[h]['cnn_common'][kind] for h in [1,2,3]]
 if kind=='f1':
  b=[results[h]['common']['fixed_05_detection']['f1'] for h in [1,2,3]];t=[results[h]['common']['detection']['f1'] for h in [1,2,3]];ax.bar(xx-.25,cnn,.25,label='CNN threshold .5');ax.bar(xx,b,.25,label='BiLSTM threshold .5');ax.bar(xx+.25,t,.25,label='BiLSTM validation threshold');ax.set_ylim(0,1.05)
 else:
  b=[results[h]['common']['magnitude']['mae'] for h in [1,2,3]];ax.bar(xx-.18,cnn,.36,label='CNN');ax.bar(xx+.18,b,.36,label='BiLSTM')
 ax.set(xticks=xx,xticklabels=['1s','2s','3s'],xlabel='Post-P horizon',ylabel=kind.upper(),title='Identical common test cohort: 1200 EQ + 1214 noise');ax.legend();save(fig,name)
checks={p:sha(Path(p))==v for p,v in cks[1]['input_hashes'].items()};assert all(checks.values())
lines=['# Standalone tabular BiLSTM synthetic EEW results','', '**Date:** 2026-11-13. Three horizon-specific models trained, validation-selected and evaluated once on frozen test inputs.','', '## Protocol','', 'Input: existing standardized feature-only HDF5, 20/30/40 completed 0.1-second bins × 7 features. First ten bins provide pre-reference context. No raw waveforms, catalog history, IDs, labels, magnitude targets or arrival metadata are model features. Existing feature scalers are reused without refitting. BiLSTM processes only the selected observed prefix bidirectionally.','',f"Architecture: two-layer bidirectional LSTM, hidden32/direction, interlayer dropout .2, temporal mean pool, dropout .2, 64→32 SiLU shared representation, binary-logit and magnitude heads; {cks[1]['parameters']:,} parameters. Settings: `{json.dumps(CFG)}`.",'', 'Loss: equal-weight BCE and earthquake-masked Huber(delta=1) on magnitudes standardized by eligible training-earthquake mean/population SD. Sentinel -1 never contributes. Poisoned-sentinel and empty-regression guards passed. Validation selection minimizes full-set mean BCE plus eligible-earthquake mean Huber. Epoch-end training/validation curves disable dropout. All three checkpoints and thresholds were selected before test evaluation.','', 'Threshold rule: maximize validation F1 over .10 to .90 in steps .01; ties prefer closest .5 then lower threshold. No test tuning. Fixed-.5 results are also reported to match the CNN. Ineligible rows are excluded before batching; magnitude metrics include every eligible earthquake regardless of detection outcome.','', '## Checkpoints','', '| Horizon | Best / run epochs | Val loss | Threshold | Fit seconds | Checkpoint |','|---|---:|---:|---:|---:|---|']
for h,ck in cks.items():lines.append(f"| {h}s | {ck['best_epoch']} / {len(ck['history'])} | {ck['best_validation_loss']:.7f} | {ck['threshold']:.2f} | {ck['training_seconds']:.2f} | models/bilstm_{h}s_best.pt |")
lines+=['','## Horizon-eligible test results','','PR-AUC uses average precision. Confusion matrices are actual rows/predicted columns [noise, earthquake].','', '| Horizon | N / EQ / excluded | Accuracy | Precision | Recall | F1 | ROC-AUC | PR-AUC | Confusion matrix |','|---|---|---:|---:|---:|---:|---:|---:|---|']
for h,r in results.items():
 d=r['detection'];lines.append(f"| {h}s | {r['eligible']} / {r['earthquakes']} / {r['excluded']} | "+' | '.join(f'{d[k]:.6f}' for k in ['accuracy','precision','recall','f1','roc_auc','pr_auc_ap'])+f" | {d['confusion_matrix']} |")
lines+=['','| Horizon | MAE | RMSE | R² | Signed error | Actual / predicted variance | Pred/actual SD | Slope |','|---|---:|---:|---:|---:|---|---:|---:|']
for h,r in results.items():
 m=r['magnitude'];lines.append(f"| {h}s | {m['mae']:.6f} | {m['rmse']:.6f} | {m['r2']:.6f} | {m['bias']:.6f} | {m['actual_variance']:.6f} / {m['predicted_variance']:.6f} | {m['std_ratio']:.4f} | {m['slope']:.4f} |")
lines+=['','| Horizon | Magnitude band | N | MAE | Signed error | Underprediction % |','|---|---|---:|---:|---:|---:|']
for h,r in results.items():
 for band,b in r['bands'].items():lines.append(f"| {h}s | {band} | {b['n']} | {b['mae']:.6f} | {b['bias']:.6f} | {b['underprediction_percent']:.2f} |")
lines+=['','## Identical common-cohort comparison','','Common cohort identity hash matches the audited CNN exactly: `'+reference['identity_sha256']+'`. Detection: 1,200 earthquakes and 1,214 noise; magnitude: those 1,200 earthquakes. Common metrics reuse the single test prediction pass. The two representations differ (CNN signed-log waveform versus seven standardized summaries), model sizes, budgets and threshold rules differ; these are benchmark comparisons, not isolated architecture effects.','', '| Horizon | CNN acc / F1 (.5) | BiLSTM acc / F1 (.5) | BiLSTM acc / F1 (val threshold) | CNN / BiLSTM MAE | CNN / BiLSTM RMSE | CNN / BiLSTM R² |','|---|---|---|---|---|---|---|']
for h,r in results.items():
 c=r['cnn_common'];f=r['common']['fixed_05_detection'];s=r['common']['detection'];m=r['common']['magnitude'];lines.append(f"| {h}s | {c['accuracy']:.6f} / {c['f1']:.6f} | {f['accuracy']:.6f} / {f['f1']:.6f} | {s['accuracy']:.6f} / {s['f1']:.6f} | {c['mae']:.6f} / {m['mae']:.6f} | {c['rmse']:.6f} / {m['rmse']:.6f} | {c['r2']:.6f} / {m['r2']:.6f} |")
lines+=['','## Timing, generalization and scientific limitations','']
for h,r in results.items():
 m=r['magnitude'];b=r['bands']['M>=5'];lat=r['latency_ms'];lines.append(f"- {h}s: model-only median/p95 latency {lat['median']:.4f}/{lat['p95']:.4f} ms. Prediction SD ratio {m['std_ratio']:.3f}, slope {m['slope']:.3f}; M≥5 underprediction {b['underprediction_percent']:.2f}% (n={b['n']}), bias {b['bias']:.3f}. Test-minus-validation gaps: F1 {r['validation_test_gaps']['f1_test_minus_val']:.4f}, magnitude MAE {r['validation_test_gaps']['mae_test_minus_val']:.4f}.")
lines+=['',f"Timing hardware: {cks[1]['environment']}. All horizons use batch1,25 warmups,200 synchronized forwards on preloaded tensors. Excludes feature extraction, disk IO, device transfer, trigger and acquisition; this is network-only latency, not end-to-end compute. Oracle-aligned 1/2/3-second observation duration is not measured autonomous detection delay or operational warning lead time.",'', 'Prediction SD/slope below one indicates compression. Strong-event error and low-range bias must accompany aggregate MAE; good global metrics do not establish reliable strong-event estimates. Epoch selection may stop at the cap without convergence; no extra tuning is performed. One seed, no confidence intervals or significance testing. The held-out cohort has been inspected in prior research and is not fresh confirmatory evidence.','', 'Noise-family shortcut check: UNKNOWN. Feature HDF5 contains no noise-profile labels; no families were invented or inferred from IDs, and upstream waveforms/metadata were not loaded. Previous CNN shortcut concerns can also affect this representation; absence of an audit here is not absence of shortcuts. Seven features include redundant energy/RMS information. No metadata input leakage or masking inconsistency was found by loader guards; original 94 feature checks were reused rather than rerun.','', 'Selected checkpoints reload and reproduce validation loss within 1e-6. Validation/test differences are reported above. Original per-horizon populations differ; only the identical common cohort supports direct cross-horizon/model comparisons. No operational or real-data generalization claim.','', '## Outputs and preservation','']
for name in ['training_validation_loss.png','actual_predicted_magnitude.png','residual_distribution.png','confusion_matrices.png','detection_f1_comparison.png','magnitude_mae_comparison.png']:lines.append(f'- [plots/{name}](plots/{name}) — 300 DPI PNG')
lines+=['', 'Two executed notebooks, three best checkpoints, six plots and this report only. Checkpoints retain configuration, histories, validation-selected threshold, train-only target transforms, feature scaler metadata, input hashes and environment. No duplicate dataset or per-epoch checkpoints. Original feature HDF5, validation report and CNN reference JSON hashes match before/after; other existing files were not opened for writing.','', 'Full numerical results follow for reproducibility without another results file.','', '```json',json.dumps(results,indent=2),'```','', '**STOP: no fusion, Attention, further tuning or cleanup.**']
with (OUT/'final_bilstm_eew_report.md').open('x',encoding='utf-8') as f:f.write('\n'.join(lines)+'\n')
print('Saved six plots and final report; original dependencies unchanged.')


Saved six plots and final report; original dependencies unchanged.


In [4]:
# Interpretation of existing saved metrics only: no inference, fitting or tuning.
from pathlib import Path
import json
p=Path(r'C:/Users/choud/Desktop/Minor project/bilstm_eew/bilstm_training_v1/final_bilstm_eew_report.md');text=p.read_text();r=json.loads(text.split('```json\n')[1].split('\n```')[0])
comments=['Observed results: validation-F1-selected thresholds trade specificity for recall; false-positive rates are '+', '.join(f"{h}s {100*d['detection']['confusion_matrix'][0][1]/sum(d['detection']['confusion_matrix'][0]):.2f}%" for h,d in r.items())+'. These are weak detection operating points despite F1 around 0.73.', 'All horizons show magnitude compression (SD ratios 0.355–0.516), overprediction below magnitude 3, and 93.13–97.71% underprediction among magnitude≥5 events. Small validation/test MAE gaps do not establish real-world validity.', 'On identical common samples at fixed threshold .5, BiLSTM F1 exceeds CNN at 1s but is lower at 2s and 3s; accuracy is lower at all three horizons. Common-cohort magnitude MAE differs from CNN by only about +0.0052, −0.0012 and +0.0021 at 1s/2s/3s. These small untested differences do not justify a general superiority claim. No settings were changed after observing them.']
anchor='## Outputs and preservation';assert anchor in text
p.write_text(text.replace(anchor,'\n\n'.join(comments)+'\n\n'+anchor,1),encoding='utf-8')
print('Recorded high false-positive rates, compression and mixed common-cohort comparison; no further training.')


Recorded high false-positive rates, compression and mixed common-cohort comparison; no further training.
